Data loading:

In [1]:
import json
import pandas as pd
import numpy as np
import re
from collections import Counter

def squad2_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    rows = []
    for article in squad["data"]:
        title = article["title"]
        for para_i, para in enumerate(article["paragraphs"]):
            context = para["context"]
            paragraph_id = f"{title}__{para_i}"

            for qa in para["qas"]:
                qid = qa["id"]
                question = qa["question"]
                is_impossible = qa.get("is_impossible", False)

                can_answer = "yes" if not is_impossible else "no"

                if can_answer == "yes" and len(qa.get("answers", [])) > 0:
                    answer_text = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text = "NO_ANSWER"
                    answer_start = -1

                rows.append({
                    "id": qid,
                    "title": title,
                    "paragraph_id": paragraph_id,
                    "context": context,
                    "question": question,
                    "can_answer": can_answer,
                    "answer_text": answer_text,
                    "answer_start": answer_start
                })

    return pd.DataFrame(rows)


def add_basic_features(df):
    df = df.copy()

    if "can_answer" in df.columns:
        df["y"] = (df["can_answer"].astype(str).str.lower() == "yes").astype(int)
    elif "is_impossible" in df.columns:
        df["y"] = (df["is_impossible"] == False).astype(int)
    else:
        df["y"] = (df["answer_text"] != "NO_ANSWER").astype(int)

    df["q_len_char"] = df["question"].astype(str).str.len()
    df["c_len_char"] = df["context"].astype(str).str.len()
 
    df["q_len_tok"] = df["question"].astype(str).str.split().str.len()
    df["c_len_tok"] = df["context"].astype(str).str.split().str.len()
    
    # answer len (solo se presente)
    if "answer_text" in df.columns:
        df["a_len_char"] = df["answer_text"].astype(str).str.len()
        df["a_len_tok"] = df["answer_text"].astype(str).str.split().str.len()
        df.loc[df["answer_text"] == "NO_ANSWER", ["a_len_char","a_len_tok"]] = 0
    
    return df


train_df = squad2_to_df("train_sampled.json")
val_df   = squad2_to_df("val_sampled.json")
test_df  = squad2_to_df("test_sampled.json")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_df = add_basic_features(train_df)
val_df   = add_basic_features(val_df)
test_df  = add_basic_features(test_df)

Train shape: (8001, 8)
Validation shape: (996, 8)
Test shape: (1004, 8)
